# 필터링 파라미터 검증 — 왜 이 값일 수밖에 없었는가

댓글 필터링 파이프라인의 9개 임계값·하이퍼파라미터를 실제 데이터로 스윕한다. 각 절의 목표는 그 값이 **성능 튜닝의 결과가 아니라 데이터 특성상 그 근방일 수밖에 없다**는 것을 보이는 것이다.

**데이터** — P절(파이프라인 개요)은 최종 논문 표본 **2014-2025 전체**(원천 215만 건, 논문 3.3 표와 동일)를 쓴다. A~G 파라미터 스윕은 **2014-2025 전체 표본**(독성 필터 입력 141만 건, archive recollect 중간 산출물)에서 수행한다. F·G는 실제 실행에서 산출된 stock_score·is_stock 컬럼을, D·E는 연도별 4만 건 층화표본을 쓴다. 전략 성과(γ·감성 MA, H·I)는 10년 K-FGI 파이프라인 전 구간 재산출이다.

**한계** — 감성 확률은 필터 통과 댓글에만 존재하므로, 일별 감성 안정성은 'τ를 더 낮추는(엄격하게) 방향'만 시뮬레이션한다. 'τ를 높이는(느슨하게) 방향'의 영향은 B-3(가중치 총량 비중)으로 판단한다.

## 0. 준비 및 데이터 로드
최초 실행 시 `compute()` 가 약 3~4분 소요되며 `_compute_cache.pkl` 에 캐시된다. 이후 실행은 즉시.

In [ ]:
%matplotlib inline
import filter_param_lib as L
import matplotlib.pyplot as plt
LANG = "ko"
L.use_style(LANG)

In [ ]:
tox, kept = L.load_all()

In [ ]:
R = L.compute(tox, kept)   # 최초 3~4분, 이후 캐시

## P. 필터링 파이프라인 개요  (최종 논문 표본 2014-2025)

감성 분석에 쓰이는 댓글은 **원천 수집 → 빈 댓글·중복 제거 → 1단계 정치 필터링 → 2단계 독성 필터링 → 3단계 경제 필터링**을 거친다. 아래 표·그래프는 논문 3.3의 최종 필터링 표(2014-2025 전체)와 동일한 수치이며, `10_Paper_outputs/04_투고_전략/FINAL_REVISION_MASTER_KR.md` 의 연도별 상세표에서 재산출한 것이다.

- 1단계 정치 필터링은 정치 키워드(인물·정당·사건명) 정규식 매칭으로 정치 프레임 댓글을 제거한다. 감성 모델이 '재앙·내란' 같은 표현을 시장 부정 감성으로 오해석하는 것을 막는 것이 목적이다.
- 제거량이 가장 큰 단계는 3단계 경제 필터링으로, 정치·독성이 아니어도 주식·경제와 무관한 일반 댓글(독성 통과분의 84.9%)을 걸러낸다.
- **연도별 원천 댓글 수 차이(P-6)**: ① 수집 기사 수가 2014년 3,651개 → 2018년 이후 ~7,200개(매일 섹션별 상위 10개 기사)로 안정, ② 기사당 평균 수집 댓글 수가 2016년 8개 → 2021년 31개로 증가 — 수집 상한이 아니라 금융 기사당 실제 댓글 증가(중앙값 2→23개, 동학개미·공매도 논란). 크롤러는 기사당 당일 댓글을 전량 수집(네이버 API 한도 ~200개). 2018년(+124%)과 2021년(피크 31만)이 대표적 증가 구간.
- **연도별 제거율 차이(P-7)**: ① 빈(삭제) 댓글 비율이 2017-2019년 25~37%로 급등(2018 드루킹 사건 후 네이버 댓글 개편), ② 2019년은 경제 필터 핵심 키워드가 2개뿐이라 최종 잔존율 2.4%로 최저 — DLF 사태·조국 사모펀드·일본 수출규제로 댓글이 비주식 이슈에 쏠려 주식 단어가 DF 0.5% 문턱을 못 넘음.

> 참고 — A~G 파라미터 스윕은 2014-2025 전체 표본에서 수행한다. F·G는 실제 stock_score·is_stock 컬럼, D·E는 연도별 4만 건 층화표본이다. 결론(τ·score 임계값의 평탄 구간, γ 무영향)은 표본 크기·연도 범위에 둔감하다. 전략 성과(H·I)는 10년 K-FGI 파이프라인 전 구간 재산출이다.

### P1

원천 수집 댓글 2,150,215건에서 시작한다. 빈 댓글·중복 제거로 정치 필터 입력이 1,575,268건(73.3%)이 되고, 1단계 정치 키워드 필터로 165,288건(입력의 10.5%), 2단계 독성 필터(τ=0.95)로 131,337건(정치 통과분의 9.3%), 3단계 경제 관련성 필터로 1,085,526건(독성 통과분의 84.9%)이 제거된다. 최종 경제/주식 관련 댓글은 193,117건(원천의 9.0%)이며 이 중 192,888건에 감성 점수가 산출되었다. (논문 3.3 필터링 표와 동일 수치. 연도별 상세표는 Appendix / `10_Paper_outputs/04_투고_전략/FINAL_REVISION_MASTER_KR.md` 참조.)

In [ ]:
L.make_fig("P1", R, LANG)

### P2

각 막대는 해당 단계를 통과한 뒤 남은 댓글 수. 원천 215만 → 정치 분류 입력 158만 → 정치 통과 141만 → 독성 통과 128만 → 최종 19.3만. 가장 크게 줄어드는 단계는 3단계 경제 필터링으로, 정치·독성이 아니어도 주식·경제와 무관한 일반 댓글을 대량 제거한다(독성 통과분의 84.9%).

In [ ]:
L.make_fig("P2", R, LANG)

### P3

좌: 연도별 정치 관련 댓글 비율. 2014-2016년 4~6% 수준에서 2018년 이후 10% 이상, 2025년 15.0%로 탄핵·계엄 국면에서 높아졌다. 우: 매칭된 정치 키워드 상위 16개(2014-2025 합계). 초기에는 박근혜·근혜, 후기에는 민주당·문재인·재명·이재명·윤석열·석열·탄핵이 상위를 차지한다. 이들 인물·정당·사건어가 감성 집계에서 정치 프레임 노이즈를 유발하므로 제거한다.

In [ ]:
L.make_fig("P3", R, LANG)

### P4

논문 Appendix 표. 원천 댓글은 2018년 이후 연 20만~31만 건으로 크게 늘었고, 최종 경제/주식 관련 댓글은 시장 이슈(2021 공매도·2023 이후 국장 논쟁)가 있던 해에 3만 건대로 많다. 최종 잔존율은 연 4~13% 범위이며 전체 평균 9.0%다.

In [ ]:
L.make_fig("P4", R, LANG)

### P5

각 연도 막대를 단계별 제거/유지로 분해한 것. 아래부터 최종 유지(경제/주식 관련), 경제 무관 제거, 독성 제거, 정치 제거, 빈 댓글·중복. 모든 연도에서 '경제 무관 제거'(연회색)가 가장 큰 비중을 차지한다.

In [ ]:
L.make_fig("P5", R, LANG)

### P6

크롤러는 매일 증권·금융 섹션에서 댓글 수 기준 상위 10개 기사(섹션별)를 선정하고, 각 기사의 당일 작성 댓글을 전량 수집한다(네이버 댓글 API 한도상 기사당 최대 약 200개, 이 한도에 걸리는 기사는 1% 미만). (좌) 수집 대상 기사 수는 2014년 3,651개에서 2018년 이후 약 7,200개로 늘어 안정된다 — 2014-2016년은 하루에 댓글 달린 금융 기사가 상위 10개를 채우지 못하는 날이 많아 적게 수집됐고, 2018년부터는 매일 10개(섹션별)를 모두 채운다. (우) 기사당 평균 수집 댓글 수(비어있지 않은 댓글 기준)는 2016년 8개에서 2021년 31개로 늘었다 — 수집 상한 때문이 아니라 금융 뉴스 기사당 실제 댓글이 증가했기 때문이다 (네이버 표시 댓글 수 중앙값 2개(2016) → 23개(2021)). 2020년 동학개미운동·2021년 공매도 재개 논란으로 금융 뉴스 댓글 참여가 폭증했다. 두 요인이 겹쳐 원천 댓글이 2014년 3.4만 → 2021년 31만으로 약 9배 늘었다.

In [ ]:
L.make_fig("P6", R, LANG)

### P7

(좌) 빈 댓글 비율: 2016년까지 0%에 가깝다가 2017-2019년 25~37%로 급등한 뒤 서서히 22%대로 낮아진다. 2018년 드루킹 댓글조작 사건 이후 네이버가 댓글 시스템을 대폭 개편하고 AI 클린봇을 확대하면서 삭제된 댓글이 빈 자리(유효 comment_id + 빈 본문)로 남기 시작했고, 2016년 이전의 삭제 댓글은 이미 완전 삭제되어 수집 자체가 안 된다. (우) 경제 관련성 필터가 뽑는 연도별 핵심 키워드 수: 2019년은 2개로 가장 적어 최종 잔존율이 2.4%로 최저다. 2019년 금융 뉴스 댓글이 DLF 사태·조국 사모펀드·일본 수출규제 등 비(非)주식 경제·사회 이슈에 집중되어, 주식 논의 단어들이 연도별 DF 0.5% 문턱을 넘지 못했기 때문이다. 핵심 키워드가 많아지는 2020-2021년(동학개미)은 잔존율도 10% 이상으로 회복한다.

In [ ]:
L.make_fig("P7", R, LANG)

### P8

(좌) 경제 필터가 뽑는 주요 주식 키워드의 연도별 문서빈도(DF). 점선(0.5%)이 핵심 키워드 문턱이다. 2019년에는 공매도·주식만 문턱을 넘고 금리·개미들·주가·기관은 모두 0.5% 아래로 떨어져 핵심 → 보조로 강등됐다. 통과 조건이 '점수 ≥ 10 이고 핵심 키워드 ≥ 1개'이므로, 핵심이 2개뿐이면 댓글에 '공매도'나 '주식'이 없으면 탈락한다 → 최종 잔존율 5.1%(다른 해 12~18%). (우) 2019년 크롤링된 증권·금융 뉴스의 주제 분포(댓글 수 가중). 은행·DLF 사태, 조국·검찰, 일본 수출규제, 연금·노동이 대부분이고 실제 주식·증시 기사는 소수다 — 개인 투자 담론(동학개미)은 2020년 3월 이후 시작됐고, 2019년 금융 뉴스 댓글은 주식시장 얘기가 아니었기 때문에 필터가 그만큼 많이 걸러낸 것이다.

In [ ]:
L.make_fig("P8", R, LANG)

## A. 독성 가중치 지수 γ  (현재값 = 2)

$w_i = (1 - \text{tox}_i)^{\gamma}$ — 독성이 높은 댓글일수록 감성 집계 기여도를 낮춘다.

**왜 2인가**
- 전략 성과가 γ에 무감각하다(H-1·H-2). γ 1·2·3에서 Sharpe 0.61–0.62, MDD −22.9% 고정 → 성과로 고른 값이 될 수 없다.
- 2는 감쇠 강도의 원칙적 중간값: γ=1은 경계 댓글(tox 0.5)을 0.5로만 낮추고, γ=3은 0.125로 과하게 죽인다. γ=2 → 0.25.
- 정보 손실 억제: 유효표본 비율이 γ=2에서 0.85, γ=3에서 0.84 (A-3).

### TOX

(좌) 가중 함수 w = (1 - tox)^γ 를 γ ∈ {0, 1, 2, 3, 5}에 대해 나타낸 것으로, 채택값 γ = 2에서 독성 점수 0.3, 0.5, 0.8인 댓글의 가중치는 각각 0.49, 0.25, 0.04이다(검은 점). 붉은 세로 점선은 hard drop 기준 τ = 0.95. (우) 독성 점수가 x 이상인 댓글이 일별 감성 가중치 총합에서 차지하는 비중(2014-2025년 표본). γ = 2 가중으로 인해 0.95 이상 댓글의 비중은 0.003%에 불과하며, 일별 감성 지표는 hard drop 기준 설정에 사실상 불변이다.

In [ ]:
L.make_fig("TOX", R, LANG)

### A1

독성 점수가 높을수록 그 댓글의 감성 집계 기여도(가중치)를 낮춘다. γ가 클수록 곡선이 아래로 휘어 중간 독성 댓글까지 빠르게 감쇠된다. 현재값 γ=2에서는 독성 0.3 → 가중치 0.49, 독성 0.5 → 0.25, 독성 0.8 → 0.04 이다. γ=1(직선)은 감쇠가 약하고, γ=3 이상은 정보 손실이 크다.

In [ ]:
L.make_fig("A1", R, LANG)

### A2

세로축은 각 γ로 계산한 일별 가중감성 시계열이 '가중을 전혀 안 한 경우'와 하루 평균 얼마나 다른지를 나타낸다. γ=2에서 0.013, γ=5까지 올려도 0.018 로 완만하다. 감성 점수의 범위가 [-1, 1] 임을 감안하면 이 차이는 매우 작다. 즉 γ를 어디에 두든 일별 감성지표는 거의 같다.

In [ ]:
L.make_fig("A2", R, LANG)

### A3

가중을 하면 실질적으로 몇 개의 댓글을 쓰는 효과인지를 전체 대비 비율로 나타낸 값(1에 가까울수록 손실 없음). γ=2에서 0.85, γ=3에서 0.84. 대부분의 댓글이 저독성이라 가중해도 표본 손실은 15% 안쪽이며 γ를 더 키워도 추가 손실이 작다.

In [ ]:
L.make_fig("A3", R, LANG)

### A4

위 칸: 실선은 γ=2(채택값)의 월평균 가중감성, 옅은 띠는 γ=1과 γ=3 사이 값의 범위(위 칸 축척에서는 거의 안 보일 만큼 좁다). 아래 칸: 그 폭(최대-최소)만 따로 확대해 그린 것으로, 전 구간 평균 0.0037, 최댓값도 0.0143에 그친다 — 월평균 가중감성 자체의 변동폭(약 0.6)에 비해 1% 안팎이다. γ를 1~3 사이 어디로 바꾸어도 감성지표의 움직임이 거의 달라지지 않음을 보여준다.

In [ ]:
L.make_fig("A4", R, LANG)

### A5

대부분의 댓글은 독성이 낮아 γ와 무관하게 가중치가 1 부근에 몰려 있다. γ가 커질수록 낮은 가중치(왼쪽 꼬리)의 댓글이 늘어나지만 전체에서 차지하는 비중은 작다.

In [ ]:
L.make_fig("A5", R, LANG)

## B. 독성 hard-drop 임계값 τ  (현재값 = 0.95)

`keep = tox < τ` — 임계값 이상인 댓글은 완전히 버린다.

**왜 0.95인가**
- 지표가 τ에 구조적으로 불변. γ=2 가중이 이미 고독성 댓글의 발언권을 눌러놨다. tox ≥ 0.95 댓글의 가중치 총합은 전체의 **0.003%** 뿐이다(B-3). τ를 0.90까지 낮춰도 MAD 0.0002 미만, 0.80까지 낮춰도 0.0004 미만(B-5).
- 제거 기준으로서 0.95는 보수적 상한: 모델이 95% 이상 확신하는 것만 버린다(이미 9.3% 제거, B-1).
- 더 올리면 명백한 독성이 통과: tox 0.95–0.99 구간은 정치·욕설 표현이 대부분이며 감성 신호가 아니다(B-2).

### B1

τ 이상인 댓글을 완전히 버린다. 현재값 0.95에서 8.6%(30,313건)가 제거된다. 0.90으로 낮추면 10.6%, 0.99로 올리면 5.1%. 임계값을 조금 옮겨도 제거량은 완만하게 변한다.

In [ ]:
L.make_fig("B1", R, LANG)

### B2

현재 임계값(0.95) 위쪽 구간별로 제거되는 댓글 수. 0.99–1.00 구간에 17,947건으로 가장 많다. 임계값을 0.97·0.99로 올리면 이 구간의 명백한 독성 댓글이 감성 집계에 들어오게 된다.

In [ ]:
L.make_fig("B2", R, LANG)

### B3

핵심 그림. γ=2 가중 때문에 고독성 댓글은 가중치가 거의 0이다. 독성 0.95 이상 댓글을 모두 합쳐도 전체 감성 가중치의 0.003% 에 불과하고, 0.90 이상도 0.015% 뿐이다. 따라서 τ를 어디에 두든 일별 감성지표와 전략 성과는 사실상 바뀌지 않는다.

In [ ]:
L.make_fig("B3", R, LANG)

### B4

독성 점수는 0 부근(정상)과 1 부근(확실한 독성)으로 갈리는 양극 분포다. 0.4–0.9 사이 애매한 구간은 상대적으로 희소하다. 이 때문에 임계값을 이 구간 어디에 두어도 제거 대상이 크게 달라지지 않는다.

In [ ]:
L.make_fig("B4", R, LANG)

### B5

임계값을 0.95에서 0.90까지 낮추면 MAD 0.00014, 0.80까지 낮추면 0.00032로, 더 많은 댓글을 제거해도 일별 감성지표는 [-1,1] 척도에서 무시할 만큼만 달라진다. (감성확률은 통과 댓글에만 있어 '더 엄격하게' 방향만 시뮬레이션 가능. '더 느슨하게' 방향은 B-3 의 가중치 비중으로 판단.)

In [ ]:
L.make_fig("B5", R, LANG)

### B6

2014–2017년 8.7–9.6% 수준이던 hard-drop 대상 비율이 2018–2019년 13.2–13.7%로 일시 상승했다가(강조), 2020년 이후 다시 9% 안팎으로, 2023–2025년에는 7–8%대로 낮아졌다. 전 기간 최댓값(13.7%, 2018)도 15% 미만이다.

In [ ]:
L.make_fig("B6", R, LANG)

## C. 키워드 추출 길이  (현재값 = 2~6글자)

이 값은 스윕이 아니라 **규칙의 근거**를 제시한다.

**왜 2~6인가**
- 하한 2: 1글자 한글 토큰 중 금융 관련은 '장' 하나뿐. 하한을 3으로 올리면 '기관·외인·환율·수급·시총·국장' 등 2글자 핵심어를 잃는다(C-2).
- 상한 6: 7글자 이상은 조사가 붙은 어절이며 짧은 핵심어의 부분문자열로 이미 포착된다. 상한 6 vs 8의 핵심어 집합은 동일(C-2).

### C1

막대는 길이별 토큰 등장 횟수, 빨간 선은 그 길이 토큰 중 주식·경제 패턴에 매칭되는 비율(오른쪽 축). 길이가 길수록 관련성 비율은 높아지지만 개수는 4글자 이후 급감한다. 노란 음영이 현재 추출 범위(2–6글자).

In [ ]:
L.make_fig("C1", R, LANG)

### C2

추출 길이 범위를 바꿨을 때 핵심 키워드로 뽑히는 단어 수. 2–6 과 2–8 이 동일하고(상한 확대 효과 없음), 3–6 은 크게 줄어든다(2글자 금융어 상실). 하한 1 은 2와 사실상 같다(1글자 금융어가 거의 없음).

In [ ]:
L.make_fig("C2", R, LANG)

## D. 핵심 키워드 DF 임계값  (실제 실행값 = DF 0.5%)

**왜 0.5% 부근인가**
- DF를 더 내리면 '가장·당장' 같은 일반어가 섞이고, 1% 이상으로 올리면 연도에 따라 핵심 키워드가 0~2개로 줄어 잔존율이 급감한다(D-2, 2019년 참조).
- 실제 실행(`3_final_filter.py`)은 `min_doc_ratio=0.005`(DF 0.5%)로 돌았다. F·G의 통과율(15.1%)은 이 실행에서 산출된 stock_score·is_stock 컬럼을 그대로 사용한 것이다.

### D1

문서빈도(DF, 해당 단어가 등장한 댓글 비율)가 임계값 이상이면 핵심, 그 30~100% 구간이면 보조 키워드. 임계값을 올리면 키워드가 빠르게 줄어든다(연평균 핵심 키워드: 0.5% → 약 6개, 1% → 약 2개). 1%에서는 전 연도에서 '공매도·주식·금리'만 남아 '주가·코스피·환율'이 빠진다. 실제 실행값 0.5%. (연도별 4만 건 층화표본)

In [ ]:
L.make_fig("D1", R, LANG)

### D2

DF 임계값이 통과율에 미치는 영향(층화표본 재추출 기준): 0.5% → 약 13%, 1% → 약 7%, 2% → 약 2%. 1% 이상으로 올리면 연도에 따라 핵심 키워드가 0~2개로 줄어 통과율이 급감한다. 실제 실행값은 0.5%이며, F·G의 통과율(15.1%)은 이 실행의 실제 stock_score·is_stock 컬럼(2014-2025 전체)을 사용한 값이다.

In [ ]:
L.make_fig("D2", R, LANG)

### D3

보조 키워드는 'DF 임계값의 (배수)배 ~ DF 임계값' 구간의 단어다. 현재값 0.3 은 DF 0.5% × 0.3 = 0.15% ~ 0.5% 구간. 배수를 0.1로 낮추면 보조 키워드가 급증하지만(희소어까지 포함) 통과율은 거의 변하지 않는다. 핵심 키워드가 이미 통과 여부를 결정하기 때문이며, 0.3 은 보조 키워드를 '자주 쓰이지만 핵심 미만'인 단어로 한정하는 무난한 값이다.

In [ ]:
L.make_fig("D3", R, LANG)

## F · G · E. 경제성 점수 임계값 · 핵심≥1 조건 · 10:3 가중치

점수 = 10·(핵심 키워드 수) + 3·(보조 키워드 수),  통과 = 점수 ≥ 10 **그리고** 핵심 ≥ 1개.

**왜 이렇게인가**
- **score ≥ 10** = 통과율 평탄 구간의 끝. cutoff 3·5·7·10 에서 통과율 15.1% 로 완전히 같고, 13 에서 10.6% 로 급락한다(F-1). 급락은 '핵심어를 정확히 하나 가진 댓글(점수 10)'이 잘리며 생긴다.
- **10 : 3** — 보조 점수를 2·5·10 무엇으로 바꿔도 통과율이 동일하다. 핵심 1개=10점이 이미 게이트를 넘기 때문. 보조 점수는 순위용.
- **핵심 ≥ 1개** — 이 조건이 없으면 '보조만 4개 이상·핵심 0'인 댓글 약 1,858건(통과분의 1.0%)이 통과한다.

### F1

핵심 그림. 점수 = 10·(핵심 키워드 수) + 3·(보조 키워드 수). 임계값을 3·5·7·10 으로 두면 통과율이 15.1% 로 완전히 같고(음영 구간), 13 에서 10.6% 로 급락한다. 이 급락은 '핵심 키워드를 정확히 하나만 가진 댓글(점수 10)' 이 잘려나가면서 생긴다. 10 은 그 평탄 구간의 끝이다.

In [ ]:
L.make_fig("F1", R, LANG)

### F2

점수 0(비관련)에 큰 봉우리, 점수 10(핵심 키워드 하나)에 뚜렷한 두 번째 봉우리가 있다. 임계값 10 은 이 두 번째 봉우리를 포함하고, 13 은 배제한다.

In [ ]:
L.make_fig("F2", R, LANG)

### E1

핵심 1개 = 핵심 점수가 그대로 게이트(임계값 10)를 넘는지가 통과를 결정한다. 핵심 점수가 10이면 보조 점수를 2·3·5·10 어느 것으로 바꿔도 통과율이 동일하고, 핵심 점수를 5·3 으로 낮추면 핵심어 1개로는 게이트를 못 넘어 통과율이 급락한다(각각 1.7%·0.7%). 즉 '핵심 10점'은 필수, '보조 3점'은 통과된 댓글의 순위에만 쓰이는 값이다.

In [ ]:
L.make_fig("E1", R, LANG)

### G1

점수 ≥ 10 만 요구하면 통과율 15.25%, 여기에 '핵심 키워드 최소 1개' 조건을 추가하면 15.10%. 그 차이는 '보조 키워드만 4개 이상·핵심 0개'로 점수 10을 넘긴 댓글들이며, 실측 1,858건(통과분의 1.0%)이다. 이들은 금융 핵심어 없이 사회·정치 맥락에서 '시장·투자' 등이 스치는 글이라, 이 조건이 걸러낸다.

In [ ]:
L.make_fig("G1", R, LANG)

## 전략 성과 (10년 K-FGI 파이프라인 재산출)

γ 와 감성 이동평균 기간을 바꿔 K-FGI → 포지션 → 전략수익률을 전 구간 재산출한 결과. 코드는 `downstream_strategy_sensitivity.py`, 수치는 `downstream_sensitivity.csv`.

**핵심** — γ 는 전략 성과에 무영향(H-1·H-2), 감성 이동평균 10일은 명확한 국소 최적(I-1). 즉 τ·γ 는 '근방이면 무엇이든 무방'이고, 감성 MA 는 '10일이어야 하는' 값이다.

### H1

γ 1·2·3 각각으로 댓글 → 일별 감성피처 → K-FGI → 포지션 → 전략수익률을 10년 전 구간 재산출한 결과. Sharpe 0.610 / 0.613 / 0.622, Calmar 0.36 부근으로 사실상 평평하다. γ=2 는 성과 최적값이 아니다.

In [ ]:
L.make_fig("H1", R, LANG)

### H2

MDD −22.9%, CVaR(5%) −2.13%, 하방 변동성 0.114 로 γ 1→3 에서 소수 셋째 자리까지 변화가 없다. 독성 가중치 지수는 전략의 하방위험에 영향을 주지 않는다.

In [ ]:
L.make_fig("H2", R, LANG)

### I1

sent_composite 를 5·10·20 거래일로 평활한 각 경우의 전략 성과. 10일이 Sharpe(0.61)·Calmar(0.36)·MDD(−22.9%) 모두에서 최적인 뚜렷한 국소 최적점이다. 5일은 노이즈, 20일은 반응 지연으로 나빠진다. (τ·γ 와 달리 이 값은 결과에 민감하므로 반드시 근거가 필요하다.)

In [ ]:
L.make_fig("I1", R, LANG)

## 결론 — 고정 관행값 vs 우리가 선택한 값

| 값 | 성격 | 판정 |
|---|---|---|
| τ = 0.95 | 우리 선택 | ✓ 지표·전략이 τ에 구조적 불변 (tox≥0.95 가중치 비중 0.003%) |
| γ = 2 | 우리 선택 | ✓ 성과 무영향, 감쇠 강도의 원칙적 중간값 |
| score ≥ 10 | 우리 선택 | ✓ 통과율 평탄 구간(3–10)의 끝 |
| DF ≥ 0.5% | 우리 선택 | ◐ 일반어 배제; 1%+ 는 연도별 핵심어 0~2개로 급감 |
| 핵심 10 : 보조 3 | 우리 선택 | ◐ 보조 점수는 게이트에 무관, 순위용 |
| 핵심 ≥ 1개 | 우리 선택 | ✓ 보조 키워드만으로 통과하는 소수 댓글 차단 |
| 2 ~ 6 글자 | 방법론 규칙 | ✓ 2글자 금융어 보존 / 긴 어절은 부분문자열로 포착 |
| 감성 10일 MA | 우리 선택 | ✓ Sharpe·MDD 모두에서 국소 최적 (근거 필요한 값) |
| 252일·60일·70% 분위수 | 금융 관행값 | 튜닝 대상 아님 |